X, y, fold, speaker = load_data_for_n(8)

In [1]:
from pathlib import Path

FOLD_CSV_PATH = Path("../data/Androids-Corpus/fold-lists.csv")
SEGMENTS_DIR = Path("../segments")
FEATURES_DIR = Path("../features")

In [ ]:
import pandas as pd
from pathlib import Pathimport SZD
from pathlib import Path

fold_csv_path = Path("../data/Androids-Corpus/fold-lists.csv")
raw = pd.read_csv(fold_csv_path, header=None, skiprows=2)

# Interview task fold columns: fold1=col7, fold2=col8, ..., fold5=col11
interview_cols = {1: 7, 2: 8, 3: 9, 4: 10, 5: 11}

speaker_to_fold = {}
for fold_num, col_idx in interview_cols.items():
    speakers_in_fold = raw[col_idx].dropna()
    for speaker in speakers_in_fold:
        stem = str(speaker).strip().strip("'")   # remove the literal quote characters
        speaker_to_fold[stem] = fold_num

print(f"Total speakers mapped: {len(speaker_to_fold)}")
print("Sample entries:", list(speaker_to_fold.items())[:5])

Total speakers mapped: 116
Sample entries: [('01_CF56_1', 1), ('02_CM57_2', 1), ('18_CM64_3', 1), ('20_CM51_3', 1), ('22_CF50_3', 1)]


In [3]:
import re

def speaker_from_filename(path: Path) -> str:
    return re.sub(r'_N\d+_seg\d+$', '', path.stem)

# quick test
test_path = Path("01_CF56_1_N8_seg3.npy")
print(speaker_from_filename(test_path))   # should print: 01_CF56_1

01_CF56_1


In [4]:
all_feature_files = sorted(FEATURES_DIR.glob("*/*.npy"))
parsed_speakers = set(speaker_from_filename(f) for f in all_feature_files)

fold_speakers = set(speaker_to_fold.keys())

print(f"Speakers found in feature filenames: {len(parsed_speakers)}")
print(f"Speakers found in fold-lists.csv:    {len(fold_speakers)}")
print(f"Do they match exactly?", parsed_speakers == fold_speakers)

missing_from_folds = parsed_speakers - fold_speakers
print(f"Speakers in features but no fold assigned: {missing_from_folds}")

Speakers found in feature filenames: 116
Speakers found in fold-lists.csv:    116
Do they match exactly? True
Speakers in features but no fold assigned: set()


In [5]:
import numpy as np

def load_data_for_n(n: int):
    """
    Load all feature vectors for a given N-level, along with their
    label (0=control, 1=depressed) and fold number.
    """
    X, y, fold, speaker = [], [], [], []

    for label_name, label_value in [("HC", 0), ("PT", 1)]:
        pattern = f"*_N{n}_seg*.npy"
        files = sorted((FEATURES_DIR / label_name).glob(pattern))

        for f in files:
            vec = np.load(f)
            spk = speaker_from_filename(f)

            X.append(vec)
            y.append(label_value)
            fold.append(speaker_to_fold[spk])
            speaker.append(spk)

    return np.array(X), np.array(y), np.array(fold), np.array(speaker)

In [6]:
X, y, fold, speaker = load_data_for_n(8)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y, return_counts=True))
print("Unique folds:", np.unique(fold, return_counts=True))

X shape: (928, 768)
y shape: (928,)
Unique labels: (array([0, 1]), array([416, 512]))
Unique folds: (array([1, 2, 3, 4, 5]), array([192, 184, 184, 184, 184]))


In [7]:
X, y, fold, speaker = load_data_for_n(8)

# For each speaker, check: do all their rows have the same fold number?
leakage_found = False
for spk in np.unique(speaker):
    speaker_folds = fold[speaker == spk]
    if len(np.unique(speaker_folds)) > 1:
        print(f"LEAKAGE: speaker {spk} appears in multiple folds: {np.unique(speaker_folds)}")
        leakage_found = True

if not leakage_found:
    print("No leakage: every speaker's segments all belong to exactly one fold.")

No leakage: every speaker's segments all belong to exactly one fold.


In [8]:
def get_fold_split(X, y, fold, fold_number: int):
    """
    Given loaded data and a fold number (1-5), return train/test split
    where `fold_number` is held out as the test set.
    """
    test_mask = (fold == fold_number)
    train_mask = ~test_mask

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    return X_train, y_train, X_test, y_test

In [9]:
X, y, fold, speaker = load_data_for_n(8)

X_train, y_train, X_test, y_test = get_fold_split(X, y, fold, fold_number=1)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train labels:", np.unique(y_train, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))
print("Total check:", X_train.shape[0] + X_test.shape[0], "should equal", X.shape[0])

Train shape: (736, 768) Test shape: (192, 768)
Train labels: (array([0, 1]), array([320, 416]))
Test labels: (array([0, 1]), array([96, 96]))
Total check: 928 should equal 928


In [10]:
import torch
import torch.nn as nn

class DepressionClassifier(nn.Module):
    """
    FFNN matching the paper's architecture:
    3 hidden layers (32, 64, 128 neurons), ReLU activations,
    final layer outputs a single probability (Depression vs Control).
    """
    def __init__(self, input_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [11]:
model = DepressionClassifier(input_dim=768)
sample_input = torch.tensor(X_train[:5], dtype=torch.float32)
output = model(sample_input)
print("Output shape:", output.shape)
print("Sample outputs:", output.detach().numpy().flatten())

Output shape: torch.Size([5, 1])
Sample outputs: [0.49573234 0.49378017 0.4933745  0.49401733 0.49318922]


In [12]:
import torch.optim as optim

def train_model(X_train, y_train, epochs=30, batch_size=32, lr=0.001):
    """
    Train the FFNN on one fold's training data.
    Matches the paper: Adam optimizer, binary cross-entropy, 30 epochs.
    """
    model = DepressionClassifier(input_dim=X_train.shape[1])

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()   # binary cross-entropy, matching the paper

    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape (N, 1) to match model output

    n_samples = X_tensor.shape[0]

    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(n_samples)   # shuffle each epoch
        epoch_loss = 0.0

        for i in range(0, n_samples, batch_size):
            indices = permutation[i:i+batch_size]
            batch_X, batch_y = X_tensor[indices], y_tensor[indices]

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, loss: {epoch_loss:.4f}")

    return model

In [13]:
model = train_model(X_train, y_train, epochs=30)

  Epoch 10/30, loss: 1.7804
  Epoch 20/30, loss: 0.1675
  Epoch 30/30, loss: 0.0234


In [14]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    with torch.no_grad():
        outputs = model(X_tensor).numpy().flatten()

    predictions = (outputs > 0.5).astype(int)
    accuracy = (predictions == y_test).mean()

    return accuracy, predictions, outputs

accuracy, predictions, raw_outputs = evaluate_model(model, X_test, y_test)
print(f"Test accuracy: {accuracy:.4f}")
print(f"Predicted labels distribution: {np.unique(predictions, return_counts=True)}")

Test accuracy: 0.8073
Predicted labels distribution: (array([0, 1]), array([ 89, 103]))


In [15]:
from sklearn.metrics import f1_score

def evaluate_model(model, X_test, y_test):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    with torch.no_grad():
        outputs = model(X_tensor).numpy().flatten()

    predictions = (outputs > 0.5).astype(int)
    accuracy = (predictions == y_test).mean()
    f1 = f1_score(y_test, predictions)

    return accuracy, f1


In [16]:
def run_cross_validation(n: int, epochs=30, verbose=True):
    """
    Run full 5-fold cross-validation for a given N-level.
    Trains and evaluates once per fold, returns average accuracy/F1.
    """
    X, y, fold, speaker = load_data_for_n(n)

    fold_accuracies = []
    fold_f1s = []

    for fold_number in [1, 2, 3, 4, 5]:
        X_train, y_train, X_test, y_test = get_fold_split(X, y, fold, fold_number)

        model = train_model(X_train, y_train, epochs=epochs)
        accuracy, f1 = evaluate_model(model, X_test, y_test)

        fold_accuracies.append(accuracy)
        fold_f1s.append(f1)

        if verbose:
            print(f"  Fold {fold_number}: accuracy={accuracy:.4f}, F1={f1:.4f}")

    mean_acc, std_acc = np.mean(fold_accuracies), np.std(fold_accuracies)
    mean_f1, std_f1 = np.mean(fold_f1s), np.std(fold_f1s)

    print(f"N={n} — Accuracy: {mean_acc:.4f} ± {std_acc:.4f}, F1: {mean_f1:.4f} ± {std_f1:.4f}")
    return mean_acc, std_acc, mean_f1, std_f1

In [17]:
run_cross_validation(n=8)

  Epoch 10/30, loss: 1.4253
  Epoch 20/30, loss: 0.0566
  Epoch 30/30, loss: 0.0139
  Fold 1: accuracy=0.7917, F1=0.7980
  Epoch 10/30, loss: 3.3568
  Epoch 20/30, loss: 0.4751
  Epoch 30/30, loss: 0.0339
  Fold 2: accuracy=0.9022, F1=0.9109
  Epoch 10/30, loss: 2.3977
  Epoch 20/30, loss: 0.2549
  Epoch 30/30, loss: 0.0300
  Fold 3: accuracy=0.8370, F1=0.8256
  Epoch 10/30, loss: 2.8278
  Epoch 20/30, loss: 0.1862
  Epoch 30/30, loss: 0.0192
  Fold 4: accuracy=0.7935, F1=0.8550
  Epoch 10/30, loss: 3.2943
  Epoch 20/30, loss: 1.1273
  Epoch 30/30, loss: 0.0380
  Fold 5: accuracy=0.8370, F1=0.8333
N=8 — Accuracy: 0.8322 ± 0.0402, F1: 0.8445 ± 0.0379


(0.8322463768115943,
 0.04020923014204374,
 0.8445494895663896,
 0.037852216948145306)

In [18]:
from tqdm import tqdm

def run_cross_validation(n: int, epochs=30, verbose=True):
    X, y, fold, speaker = load_data_for_n(n)

    fold_accuracies = []
    fold_f1s = []

    for fold_number in tqdm([1, 2, 3, 4, 5], desc=f"N={n} folds", leave=False):
        X_train, y_train, X_test, y_test = get_fold_split(X, y, fold, fold_number)

        model = train_model(X_train, y_train, epochs=epochs)
        accuracy, f1 = evaluate_model(model, X_test, y_test)

        fold_accuracies.append(accuracy)
        fold_f1s.append(f1)

        if verbose:
            print(f"  Fold {fold_number}: accuracy={accuracy:.4f}, F1={f1:.4f}")

    mean_acc, std_acc = np.mean(fold_accuracies), np.std(fold_accuracies)
    mean_f1, std_f1 = np.mean(fold_f1s), np.std(fold_f1s)

    print(f"N={n} — Accuracy: {mean_acc:.4f} ± {std_acc:.4f}, F1: {mean_f1:.4f} ± {std_f1:.4f}")
    return mean_acc, std_acc, mean_f1, std_f1

In [19]:
results = {}

for n in tqdm([1, 2, 4, 8, 16, 32, 64], desc="N-levels"):
    print(f"\n=== N={n} ===")
    mean_acc, std_acc, mean_f1, std_f1 = run_cross_validation(n=n, verbose=False)
    results[n] = {"acc_mean": mean_acc, "acc_std": std_acc, "f1_mean": mean_f1, "f1_std": std_f1}

print("\n\nSummary:")
for n, r in results.items():
    print(f"N={n:2d} — Accuracy: {r['acc_mean']:.4f} ± {r['acc_std']:.4f}, F1: {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")

N-levels:   0%|          | 0/7 [00:00<?, ?it/s]


=== N=1 ===


  Epoch 10/30, loss: 1.3209
  Epoch 20/30, loss: 0.2167
  Epoch 30/30, loss: 0.0414
  Epoch 10/30, loss: 1.5786
  Epoch 20/30, loss: 0.3677
  Epoch 30/30, loss: 0.0775
  Epoch 10/30, loss: 1.2644
  Epoch 20/30, loss: 0.3177


N-levels:  14%|█▍        | 1/7 [00:00<00:01,  3.54it/s]

  Epoch 30/30, loss: 0.0760
  Epoch 10/30, loss: 1.1053
  Epoch 20/30, loss: 0.2128
  Epoch 30/30, loss: 0.0363
  Epoch 10/30, loss: 1.3878
  Epoch 20/30, loss: 0.3051
  Epoch 30/30, loss: 0.0698
N=1 — Accuracy: 0.8880 ± 0.0440, F1: 0.9026 ± 0.0310

=== N=2 ===


...

  Epoch 30/30, loss: 0.0791
  Epoch 10/30, loss: 25.6299
  Epoch 20/30, loss: 3.7375


  Epoch 30/30, loss: 0.1061
  Epoch 10/30, loss: 19.2892
  Epoch 20/30, loss: 8.8712


  Epoch 30/30, loss: 0.0172
  Epoch 10/30, loss: 29.6292
  Epoch 20/30, loss: 7.0894


N-levels: 100%|██████████| 7/7 [00:21<00:00,  3.04s/it]

  Epoch 30/30, loss: 0.0600
N=64 — Accuracy: 0.7453 ± 0.0587, F1: 0.7637 ± 0.0397


Summary:
N= 1 — Accuracy: 0.8880 ± 0.0440, F1: 0.9026 ± 0.0310
N= 2 — Accuracy: 0.8924 ± 0.0383, F1: 0.9026 ± 0.0211
N= 4 — Accuracy: 0.8557 ± 0.0422, F1: 0.8708 ± 0.0341
N= 8 — Accuracy: 0.8269 ± 0.0441, F1: 0.8402 ± 0.0424
N=16 — Accuracy: 0.7947 ± 0.0683, F1: 0.8120 ± 0.0429
N=32 — Accuracy: 0.7688 ± 0.0582, F1: 0.7862 ± 0.0370
N=64 — Accuracy: 0.7453 ± 0.0587, F1: 0.7637 ± 0.0397


In [20]:
def get_fold_split(X, y, fold, speaker, fold_number: int):
    test_mask = (fold == fold_number)
    train_mask = ~test_mask

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test, speaker_test = X[test_mask], y[test_mask], speaker[test_mask]

    return X_train, y_train, X_test, y_test, speaker_test

In [21]:
def evaluate_model(model, X_test, y_test, speaker_test):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    with torch.no_grad():
        outputs = model(X_tensor).numpy().flatten()

    seg_preds = (outputs > 0.5).astype(int)

    # segment-level (what we had before)
    seg_acc = (seg_preds == y_test).mean()
    seg_f1 = f1_score(y_test, seg_preds)

    # recording-level: majority vote per speaker
    rec_true, rec_pred = [], []
    for spk in np.unique(speaker_test):
        mask = (speaker_test == spk)
        true_label = y_test[mask][0]              # all segments share the same true label
        vote = seg_preds[mask].mean() > 0.5        # fraction predicted "Depression" > 50%
        rec_true.append(true_label)
        rec_pred.append(int(vote))

    rec_true, rec_pred = np.array(rec_true), np.array(rec_pred)
    rec_acc = (rec_pred == rec_true).mean()
    rec_f1 = f1_score(rec_true, rec_pred)

    return seg_acc, seg_f1, rec_acc, rec_f1

In [22]:
def run_cross_validation(n: int, epochs=30, verbose=True):
    X, y, fold, speaker = load_data_for_n(n)

    seg_accs, seg_f1s, rec_accs, rec_f1s = [], [], [], []

    for fold_number in tqdm([1, 2, 3, 4, 5], desc=f"N={n} folds", leave=False):
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(X, y, fold, speaker, fold_number)

        model = train_model(X_train, y_train, epochs=epochs)
        seg_acc, seg_f1, rec_acc, rec_f1 = evaluate_model(model, X_test, y_test, speaker_test)

        seg_accs.append(seg_acc); seg_f1s.append(seg_f1)
        rec_accs.append(rec_acc); rec_f1s.append(rec_f1)

        if verbose:
            print(f"  Fold {fold_number}: seg_acc={seg_acc:.4f}, seg_f1={seg_f1:.4f}, rec_acc={rec_acc:.4f}, rec_f1={rec_f1:.4f}")

    result = {
        "seg_acc_mean": np.mean(seg_accs), "seg_acc_std": np.std(seg_accs),
        "seg_f1_mean": np.mean(seg_f1s), "seg_f1_std": np.std(seg_f1s),
        "rec_acc_mean": np.mean(rec_accs), "rec_acc_std": np.std(rec_accs),
        "rec_f1_mean": np.mean(rec_f1s), "rec_f1_std": np.std(rec_f1s),
    }

    print(f"N={n} — Recording-level: Acc={result['rec_acc_mean']:.4f}±{result['rec_acc_std']:.4f}, F1={result['rec_f1_mean']:.4f}±{result['rec_f1_std']:.4f}")
    return result

In [23]:
results = {}

for n in tqdm([1, 2, 4, 8, 16, 32, 64], desc="N-levels"):
    print(f"\n=== N={n} ===")
    results[n] = run_cross_validation(n=n, verbose=False)

print("\n\nSummary (recording-level, comparable to the paper):")
for n, r in results.items():
    print(f"N={n:2d} — Accuracy: {r['rec_acc_mean']:.4f} ± {r['rec_acc_std']:.4f}, F1: {r['rec_f1_mean']:.4f} ± {r['rec_f1_std']:.4f}")

N-levels:   0%|          | 0/7 [00:00<?, ?it/s]


=== N=1 ===


  Epoch 10/30, loss: 0.9545
  Epoch 20/30, loss: 0.1147
  Epoch 30/30, loss: 0.0179
  Epoch 10/30, loss: 1.4535
  Epoch 20/30, loss: 0.2979
  Epoch 30/30, loss: 0.0483
  Epoch 10/30, loss: 1.3368
  Epoch 20/30, loss: 0.2958
  Epoch 30/30, loss: 0.0663
  Epoch 10/30, loss: 1.1267


N-levels:  14%|█▍        | 1/7 [00:00<00:01,  3.74it/s]

  Epoch 20/30, loss: 0.2176
  Epoch 30/30, loss: 0.0363
  Epoch 10/30, loss: 1.3875
  Epoch 20/30, loss: 0.3029
  Epoch 30/30, loss: 0.0847
N=1 — Recording-level: Acc=0.9141±0.0377, F1=0.9247±0.0284

=== N=2 ===


...

  Epoch 30/30, loss: 6.4189
  Epoch 10/30, loss: 23.0280
  Epoch 20/30, loss: 7.4287


  Epoch 30/30, loss: 4.1768
  Epoch 10/30, loss: 22.7514
  Epoch 20/30, loss: 6.0772


  Epoch 30/30, loss: 3.0367
  Epoch 10/30, loss: 26.0289
  Epoch 20/30, loss: 2.8017


N-levels: 100%|██████████| 7/7 [00:21<00:00,  3.09s/it]

  Epoch 30/30, loss: 2.8562
N=64 — Recording-level: Acc=0.8109±0.1037, F1=0.8237±0.0816


Summary (recording-level, comparable to the paper):
N= 1 — Accuracy: 0.9141 ± 0.0377, F1: 0.9247 ± 0.0284
N= 2 — Accuracy: 0.8880 ± 0.0440, F1: 0.8932 ± 0.0396
N= 4 — Accuracy: 0.8964 ± 0.0592, F1: 0.9062 ± 0.0394
N= 8 — Accuracy: 0.9058 ± 0.0622, F1: 0.9146 ± 0.0512
N=16 — Accuracy: 0.8801 ± 0.0990, F1: 0.8938 ± 0.0810
N=32 — Accuracy: 0.8192 ± 0.0740, F1: 0.8210 ± 0.0612
N=64 — Accuracy: 0.8109 ± 0.1037, F1: 0.8237 ± 0.0816


In [ ]:
MODELS_DIR = Path("../Models/Baseline_model_v2")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Created:", MODELS_DIR)

Created: ../Models/Baseline_model


In [25]:
def run_cross_validation(n: int, epochs=30, verbose=True, save_models=True):
    X, y, fold, speaker = load_data_for_n(n)

    seg_accs, seg_f1s, rec_accs, rec_f1s = [], [], [], []

    for fold_number in tqdm([1, 2, 3, 4, 5], desc=f"N={n} folds", leave=False):
        X_train, y_train, X_test, y_test, speaker_test = get_fold_split(X, y, fold, speaker, fold_number)

        model = train_model(X_train, y_train, epochs=epochs)
        seg_acc, seg_f1, rec_acc, rec_f1 = evaluate_model(model, X_test, y_test, speaker_test)

        seg_accs.append(seg_acc); seg_f1s.append(seg_f1)
        rec_accs.append(rec_acc); rec_f1s.append(rec_f1)

        if save_models:
            model_path = MODELS_DIR / f"N{n}_fold{fold_number}.pt"
            torch.save(model.state_dict(), model_path)

        if verbose:
            print(f"  Fold {fold_number}: seg_acc={seg_acc:.4f}, seg_f1={seg_f1:.4f}, rec_acc={rec_acc:.4f}, rec_f1={rec_f1:.4f}")

    result = {
        "seg_acc_mean": np.mean(seg_accs), "seg_acc_std": np.std(seg_accs),
        "seg_f1_mean": np.mean(seg_f1s), "seg_f1_std": np.std(seg_f1s),
        "rec_acc_mean": np.mean(rec_accs), "rec_acc_std": np.std(rec_accs),
        "rec_f1_mean": np.mean(rec_f1s), "rec_f1_std": np.std(rec_f1s),
    }

    print(f"N={n} — Recording-level: Acc={result['rec_acc_mean']:.4f}±{result['rec_acc_std']:.4f}, F1={result['rec_f1_mean']:.4f}±{result['rec_f1_std']:.4f}")
    return result

In [26]:
results = {}

for n in tqdm([1, 2, 4, 8, 16, 32, 64], desc="N-levels"):
    print(f"\n=== N={n} ===")
    results[n] = run_cross_validation(n=n, verbose=False, save_models=True)

print("\n\nFiles saved:")
for f in sorted(MODELS_DIR.glob("*.pt")):
    print(" ", f.name)

N-levels:   0%|          | 0/7 [00:00<?, ?it/s]


=== N=1 ===


  Epoch 10/30, loss: 1.4174
  Epoch 20/30, loss: 0.2286
  Epoch 30/30, loss: 0.0281
  Epoch 10/30, loss: 1.4533
  Epoch 20/30, loss: 0.3080
  Epoch 30/30, loss: 0.0630
  Epoch 10/30, loss: 1.1778
  Epoch 20/30, loss: 0.3131
  Epoch 30/30, loss: 0.0606


N-levels:  14%|█▍        | 1/7 [00:00<00:01,  3.67it/s]

  Epoch 10/30, loss: 1.1239
  Epoch 20/30, loss: 0.2708
  Epoch 30/30, loss: 0.0602
  Epoch 10/30, loss: 1.3611
  Epoch 20/30, loss: 0.3541
  Epoch 30/30, loss: 0.0788
N=1 — Recording-level: Acc=0.8880±0.0587, F1=0.9027±0.0395

=== N=2 ===


...

  Epoch 30/30, loss: 2.4374
  Epoch 10/30, loss: 25.8288
  Epoch 20/30, loss: 8.2893


  Epoch 30/30, loss: 2.1857
  Epoch 10/30, loss: 24.4578
  Epoch 20/30, loss: 5.3410


  Epoch 30/30, loss: 4.5244
  Epoch 10/30, loss: 22.0800
  Epoch 20/30, loss: 10.7509


N-levels: 100%|██████████| 7/7 [00:20<00:00,  2.91s/it]

  Epoch 30/30, loss: 0.0257
N=64 — Recording-level: Acc=0.8022±0.1176, F1=0.8104±0.0989


Files saved:
  N16_fold1.pt
  N16_fold2.pt
  N16_fold3.pt
  N16_fold4.pt
  N16_fold5.pt
  N1_fold1.pt
  N1_fold2.pt
  N1_fold3.pt
  N1_fold4.pt
  N1_fold5.pt
  N2_fold1.pt
  N2_fold2.pt
  N2_fold3.pt
  N2_fold4.pt
  N2_fold5.pt
  N32_fold1.pt
  N32_fold2.pt
  N32_fold3.pt
  N32_fold4.pt
  N32_fold5.pt
  N4_fold1.pt
  N4_fold2.pt
  N4_fold3.pt
  N4_fold4.pt
  N4_fold5.pt
  N64_fold1.pt
  N64_fold2.pt
  N64_fold3.pt
  N64_fold4.pt
  N64_fold5.pt
  N8_fold1.pt
  N8_fold2.pt
  N8_fold3.pt
  N8_fold4.pt
  N8_fold5.pt


In [27]:
import json

results_serializable = {str(n): r for n, r in results.items()}
with open(MODELS_DIR / "results_summary.json", "w") as f:
    json.dump(results_serializable, f, indent=2)

print("Saved results_summary.json")

Saved results_summary.json


In [2]:
import json
from pathlib import Path

MODELS_DIR = Path("../Models/Baseline_model")

with open(MODELS_DIR / "results_summary.json") as f:
    saved_results = json.load(f)

print("Summary (recording-level), loaded from disk:")
for n, r in saved_results.items():
    print(f"N={n:>2} — Accuracy: {r['rec_acc_mean']:.4f} ± {r['rec_acc_std']:.4f}, F1: {r['rec_f1_mean']:.4f} ± {r['rec_f1_std']:.4f}")

Summary (recording-level), loaded from disk:
N= 1 — Accuracy: 0.8880 ± 0.0587, F1: 0.9027 ± 0.0395
N= 2 — Accuracy: 0.8797 ± 0.0685, F1: 0.8879 ± 0.0559
N= 4 — Accuracy: 0.8880 ± 0.0804, F1: 0.9003 ± 0.0580
N= 8 — Accuracy: 0.8888 ± 0.0680, F1: 0.8993 ± 0.0584
N=16 — Accuracy: 0.8623 ± 0.1176, F1: 0.8747 ± 0.0960
N=32 — Accuracy: 0.8196 ± 0.0824, F1: 0.8329 ± 0.0665
N=64 — Accuracy: 0.8022 ± 0.1176, F1: 0.8104 ± 0.0989
